In [1]:
import numpy as np
import torch
import napari
from tqdm import tqdm
import anndata
import matplotlib.colors
import matplotlib.pyplot as plt
from ATLAS.Analysis.Classification import *
from scipy.interpolate import interp1d


In [ ]:
from ATLAS.Registration.execute import *
adata = anndata.read_h5ad("/scratchdata1/Images2024/Zach/MouseBrainAtlas/WTM01_3.2.A_2.2.B_1.2.C_6.2.D_5.2.E_4.2.F_2024Apr08/Processing_2025Feb04/WellA-Section3/anndata/anndata.h5ad")
animal = 'WTM01'
adata.obs['animal'] = animal
adata.obs['dataset'] = 'WTM01_3.2.A_2.2.B_1.2.C_6.2.D_5.2.E_4.2.F_2024Apr08'
adata.obs['processing'] = 'Processing_2025Feb04'
registration_path = '/scratchdata1/Images2024/Zach/MouseBrainAtlas/WTM01_3.2.A_2.2.B_1.2.C_6.2.D_5.2.E_4.2.F_2024Apr08/Registration_2024Jul02'
section_acq_name = 'WellA-Section3'
XYZC  = Registration_Class(adata.copy(),registration_path,section_acq_name,verbose=False,regularize=True).run()
adata.obs['ccf_x'] = XYZC['ccf_x']
adata.obs['ccf_y'] = XYZC['ccf_y']
adata.obs['ccf_z'] = XYZC['ccf_z']
adata.obs['old_section_name'] = section_acq_name
adata.obs['registration_path'] = registration_path
section_name = f"{animal}_{adata.obs['ccf_x'].mean():.1f}"
adata.obs['section_name'] = section_name
adata.obs['Slice'] = section_name
from ATLAS.Utils import geomu
XY = np.array(adata.obs[["ccf_z","ccf_y"]])

adata.obs['in_large_comp'] = geomu.in_graph_large_connected_components(XY,Section = None,max_dist = 0.05,large_comp_def = 0.1,plot_comp = False)
adata = adata[adata.obs['in_large_comp']==True].copy()
print(f"Keeping {adata.shape[0]} cells after component filtering for {section_acq_name}")
from ATLAS.Utils import basicu
n_cells_pre_nuc_filtering = adata.shape[0]
adata.layers['nuc_mask'] = basicu.filter_cells_nuc(adata)
adata = adata[np.sum(adata.layers['nuc_mask']==False,axis=1)<2].copy()
adata = adata[np.clip(np.array(adata.layers['raw']).copy().sum(1),1,None)>100].copy()
print(f"Keeping {adata.shape[0]} cells after nuc filtering for {section_acq_name}")
np.random.seed(42)
self = SingleCellAlignmentLeveragingExpectations(adata,visualize=False,verbose=False)
self.likelihood_only = False
self.calculate_spatial_priors()
self.load_reference()
self.model = LogisticRegression(max_iter=1000,random_state=42) 
self.supervised_neuron_annotation()

self.supervised_harmonization()
self.determine_neighbors()
import matplotlib.pyplot as plt
adata_updated = self.measured.copy()
c = adata_updated.obs['subclass_color']
plt.figure(figsize=(15,15))
plt.scatter(adata_updated.obs['ccf_z'],adata.obs['ccf_y'],c=c,s=2,marker=',', edgecolors='none', linewidths=0)
plt.grid('off')
plt.axis('off')
plt.show()
adata = adata_updated.copy()

In [ ]:
adata_updated

In [ ]:
adata_updated.write('/scratchdata1/Images2024/Zach/MouseBrainAtlas/WTM01_3.2.A_2.2.B_1.2.C_6.2.D_5.2.E_4.2.F_2024Apr08/Processing_2025Feb04/WellA-Section3/anndata/classified_adata.h5ad')

In [5]:
adata = adata_updated.copy()

In [ ]:
imputation_cells = np.unique(adata.obs[[i for i in adata.obs.columns if 'neighbor' in i]].values)
imputation_cells = np.unique([i.split('raise')[0] for i in imputation_cells])
imputation_cells.shape

In [ ]:
imputation_cells

In [ ]:
import gc
imputation_reference_adata = []
# V3
data_path = '/bluedata/ExternalData/Allen_WMB_2024Mar06/expression_matrices/WMB-10Xv3/20230630'
for file in tqdm(os.listdir(data_path)):
    if 'raw.h5ad' in file:
        temp_adata = anndata.read_h5ad(os.path.join(data_path,file))
        shared_cells = np.intersect1d(imputation_cells,temp_adata.obs_names)
        imputation_reference_adata.append(temp_adata[shared_cells].copy())
        del temp_adata
        gc.collect()

#V2
data_path = '/bluedata/ExternalData/Allen_WMB_2024Mar06/expression_matrices/WMB-10Xv2/20230630/'
for file in tqdm(os.listdir(data_path)):
    if 'raw.h5ad' in file:
        temp_adata = anndata.read_h5ad(os.path.join(data_path,file))
        shared_cells = np.intersect1d(imputation_cells,temp_adata.obs_names)
        imputation_reference_adata.append(temp_adata[shared_cells].copy())
        del temp_adata
        gc.collect()
imputation_reference_adata = anndata.concat(imputation_reference_adata)
gc.collect()


In [ ]:
imputation_reference_adata.write('/scratchdata1/Images2024/Zach/MouseBrainAtlas/WTM01_3.2.A_2.2.B_1.2.C_6.2.D_5.2.E_4.2.F_2024Apr08/Processing_2025Feb04/WellA-Section3/anndata/imputation_reference.h5ad')

In [ ]:
neighbor_columns = [i for i in adata.obs.columns if 'neighbor' in i]
imputed = None
for i,column in tqdm(enumerate(neighbor_columns)):
    neighbors = [i.split('raise')[0] for i in adata.obs[column]]
    neighbor = imputation_reference_adata[neighbors,:].X.toarray().astype(float)
    if isinstance(imputed,type(None)):
        imputed = neighbor
    else:
        imputed = imputed + neighbor
imputed = imputed/len(neighbor_columns)
# imputed_adata = anndata.AnnData(X=imputed,obs=adata.obs,var=pd.DataFrame(index=imputation_reference.var.index))
imputed_adata = anndata.AnnData(X=imputed,obs=adata.obs,var=imputation_reference_adata.var)
imputed_adata.write('/scratchdata1/Images2024/Zach/MouseBrainAtlas/WTM01_3.2.A_2.2.B_1.2.C_6.2.D_5.2.E_4.2.F_2024Apr08/Processing_2025Feb04/WellA-Section3/anndata/imputed_adata.h5ad')